# Local English text-to-speech with SpeechT5

Install `requirements-local-tts.txt`. This example uses a supplied speaker embedding, not a user-trained or cloned voice. Initial model downloads require internet; model weights are not bundled. Outputs are cleared and model inference has not been rerun.

In [ ]:
from pathlib import Path
import sys

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "ai_lab.py").is_file()), None)
if ROOT is None:
    raise RuntimeError("Open this notebook from inside the repository folder.")
sys.path.insert(0, str(ROOT))
from ai_lab import chat, image_messages, transcribe_audio, synthesize_speech

In [ ]:
import torch
import soundfile as sf
import json
from transformers import pipeline
from IPython.display import Audio, display
speaker_data = json.loads((ROOT / "04_text_to_speech/voicedataset/speaker_embedding.json").read_text())
speaker_embedding = torch.tensor(speaker_data["xvector"], dtype=torch.float32).unsqueeze(0)
assert speaker_embedding.shape == (1, 512)
synthesizer = pipeline("text-to-speech", model="microsoft/speecht5_tts", device=-1)

In [ ]:
speech = synthesizer("Hello, I am your assistant! What is your name?",
    forward_params={"speaker_embeddings": speaker_embedding})
output = ROOT / "generated/local_speecht5.wav"
output.parent.mkdir(parents=True, exist_ok=True)
sf.write(output, speech["audio"], samplerate=speech["sampling_rate"])
display(Audio(speech["audio"], rate=speech["sampling_rate"]))
print(output.name)

Speaker vectors come from [Matthijs/cmu-arctic-xvectors](https://huggingface.co/datasets/Matthijs/cmu-arctic-xvectors). The model is [microsoft/speecht5_tts](https://huggingface.co/microsoft/speecht5_tts). See the source cards for licensing and intended use. The audio samples already included in this folder are supplied examples, not outputs of this run.